# 6 · Intertidal elevation map

### From Landsat time series to a height (m) for every intertidal pixel

Page 5 explained **NDWI** — the wet/dry index that links satellite reflectance to tide height.
This page goes one step further: it estimates **sediment height
(m, MSL)** on the intertidal flat using the production DEA algorithm
`intertidal.elevation()`.

**What this notebook answers**

1. Do we have **enough Landsat scenes** over several years?
2. Which **tide levels** do those scenes actually sample (same plot style as page 3)?
3. What does the **elevation map** look like, and where is the fit **uncertain**?

**Pipeline (one study area)**

| Step | Action |
|---|---|
| 5 | Download Landsat → NDWI stack (year caches) |
| 6 | Tide × satellite plot (FES2022 + scene dots) |
| 7 | Run `elevation()` — fit height per pixel |
| 8 | Map elevation + uncertainty |
| 9 | QA layers (NDWI correlation, wet frequency, clear count) |
| 10 | Export NetCDF / GeoTIFF |

**Key outputs**

| Variable | Meaning |
|---|---|
| `elevation` | Sediment height (m, MSL) on intertidal flats |
| `elevation_uncertainty` | Fit uncertainty (m) — **lower is better** |
| `qa_ndwi_freq` | Fraction of clear observations that were wet |
| `qa_count_clear` | Number of clear observations per pixel |

**Rule of thumb:** use a **3–4 year** window and **≥ 20 clear Landsat scenes**.
Always read Step 6 before trusting the map.

> **Heavy step:** first run downloads GB of Landsat data; Step 7 loads everything
> into memory. Keep `DELTA` modest (~0.08° ≈ 8 km) while learning.

**Previous:** [5 · NDWI](05_ndwi.ipynb)


## Step 1 — Imports

Libraries for this page:

- **`pystac_client` + `planetary_computer`** — search and sign Landsat STAC items (page 2).
- **`odc.stac`** — load reflectance bands into lazy xarray/Dask arrays.
- **`dea_tools.dask`** — local Dask cluster for parallel download/resampling.
- **`intertidal.elevation`** — DEA production elevation algorithm.
- **`cache_utils`** — shared FES2022 tide cache + datetime normalisation (pages 3–4).
- **`rioxarray`** — GeoTIFF export in Step 10.


In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import odc.stac
import planetary_computer
import pystac_client
import rioxarray  # noqa: F401 — registers .rio accessor on xarray

from dea_tools.dask import create_local_dask_cluster
from intertidal.elevation import elevation
from cache_utils import load_or_compute_tides, _utc_datetime_ns

warnings.filterwarnings("ignore")
print("Imports OK.")


## Step 2 — Configuration

Use the **same site centre** as pages 3–4 (`LON`, `LAT`, `SITE_NAME`).

Unlike page 5 (NDWI demo scenes), elevation needs **Landsat only** (30 m, long archive) and a
**multi-year** window so each pixel sees many different tide levels.

| Variable | Role |
|---|---|
| `DELTA` | Half-width of the box in degrees (~0.08° ≈ 8 km side length) |
| `START` / `END` | Analysis window — 3–4 years recommended |
| `MAX_CLOUD` | Scene-level cloud ceiling passed to STAC search |
| `MIN_SCENES_RELIABLE` | Printed warning if total scenes fall below this |
| `OVERWRITE` | `True` → re-download a year even if Zarr cache exists |
| `TIDE_DIR` / `TIDE_MODEL` | FES2022 constituent files (same as pages 3–4) |


In [ ]:
# === EDIT THESE ===
LON        = 4.81050
LAT        = 52.98886
SITE_NAME  = "WaddenSea"

DELTA      = 0.08
START      = "2023-01-01"
END        = "2025-12-31"

TIDE_DIR   = "./tide_models"
TIDE_MODEL = "FES2022"

RESOLUTION = 30          # metres — Landsat native ground sampling
MAX_CLOUD  = 80          # percent — scene-level STAC filter
MIN_SCENES_RELIABLE = 20 # warn if fewer clear-scene passes in window
OVERWRITE  = False       # True rebuilds ndwi_<year>.zarr caches

# --- derived paths (usually no need to edit) ---
os.environ["EO_TIDES_TIDE_MODELS"] = TIDE_DIR
BBOX = (LON - DELTA, LAT - DELTA, LON + DELTA, LAT + DELTA)
CACHE_DIR = Path("cache") / f"elevation_{SITE_NAME}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
YEARS = list(range(int(START[:4]), int(END[:4]) + 1))

print(f"Site    : {SITE_NAME}  ({LAT:.4f} N, {LON:.4f} E)")
print(f"AOI     : {BBOX}")
print(f"Window  : {START} to {END}  ({len(YEARS)} calendar years)")
print(f"Cache   : {CACHE_DIR}")


## Step 3 — Start local Dask cluster

`odc.stac.load()` returns **lazy** Dask arrays. A local cluster lets your machine use
multiple CPU cores while scenes are downloaded and resampled to 30 m UTM.

You should see a `<Client: ...>` line when the cluster starts.


In [ ]:
create_local_dask_cluster()
print("Dask cluster ready.")


## Step 4 — Helper functions

These functions wrap the Landsat → NDWI pipeline. They are defined here so the steps
below stay short and readable.

| Function | Purpose |
|---|---|
| `load_landsat_ndwi` | STAC search → reflectance → NDWI (+ NDVI) |
| `cache_year` | Download one calendar year and save to Zarr |
| `load_scene_table` | Merge per-year scene metadata CSVs |
| `tag_scenes_with_tide` | Attach FES2022 height to each scene time |


In [ ]:
PC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"


def load_landsat_ndwi(start: str, end: str):
    """Download Landsat C2 L2 scenes and return an NDWI (+ NDVI) xarray stack."""
    catalog = pystac_client.Client.open(PC_URL, modifier=planetary_computer.sign_inplace)

    # Tier-1 Landsat 5/7/8/9 only; filter cloudy scenes at catalogue level
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=BBOX,
        datetime=f"{start}/{end}",
        query={
            "eo:cloud_cover": {"lt": MAX_CLOUD},
            "platform": {"in": ["landsat-5", "landsat-7", "landsat-8", "landsat-9"]},
            "landsat:collection_category": {"in": ["T1"]},
        },
    )
    items = search.item_collection()
    if len(items) == 0:
        return None, items

    # Load to UTM grid; groupby solar_day merges same-day paths
    ds = odc.stac.load(
        items,
        bands=["red", "green", "nir08", "qa_pixel"],
        bbox=BBOX,
        crs="utm",
        resolution=RESOLUTION,
        chunks={"x": 2048, "y": 2048},
        groupby="solar_day",
        resampling={"qa_pixel": "nearest", "*": "cubic"},
        fail_on_error=False,
    )

    # Landsat Collection 2 QA: bit 3 = cloud, bit 4 = shadow → keep clear pixels
    cloud_or_shadow = (
        np.bitwise_and(ds.qa_pixel, 1 << 3) | np.bitwise_and(ds.qa_pixel, 1 << 4)
    )
    ds = ds.where(cloud_or_shadow == 0).drop_vars("qa_pixel")

    # USGS scale/offset for surface reflectance; clip to [0, 1]
    ds = (ds.where(ds != 0) * 0.0000275 - 0.2).clip(0, 1)

    # NDWI = wet/dry indicator used by elevation(); NDVI kept for optional QC
    ndwi = ((ds.green - ds.nir08) / (ds.green + ds.nir08)).rename("ndwi")
    ndvi = ((ds.nir08 - ds.red) / (ds.nir08 + ds.red)).rename("ndvi")
    return xr.Dataset({"ndwi": ndwi, "ndvi": ndvi}), items


def cache_year(year: int, overwrite: bool = False):
    """Cache one calendar year of NDWI to Zarr; return (path, n_scenes)."""
    zpath = CACHE_DIR / f"ndwi_{year}.zarr"
    meta = CACHE_DIR / f"scenes_{year}.csv"

    if zpath.exists() and not overwrite:
        n = len(pd.read_csv(meta)) if meta.exists() else np.nan
        print(f"  {year}: cache hit ({int(n) if n == n else '?'} scenes)")
        return zpath, n

    print(f"  {year}: downloading Landsat ...")
    ds, items = load_landsat_ndwi(f"{year}-01-01", f"{year}-12-31")
    n = len(items)
    if ds is None or n == 0:
        print(f"  {year}: no scenes")
        return None, 0

    # Scene metadata for the tide × satellite plot (Step 6)
    pd.DataFrame({
        "time": [it.properties.get("datetime") for it in items],
        "platform": [it.properties.get("platform", "?") for it in items],
        "cloud_cover": [it.properties.get("eo:cloud_cover", np.nan) for it in items],
    }).to_csv(meta, index=False)

    # Materialise time dimension for stable Zarr writes
    ds = ds.chunk({"time": -1, "x": 2048, "y": 2048})
    ds.to_zarr(zpath, mode="w")
    print(f"  {year}: cached {n} scenes → {zpath.name}")
    return zpath, n


def load_scene_table() -> pd.DataFrame:
    """Combine scenes_<year>.csv files written during Step 5."""
    frames = []
    for year in YEARS:
        meta = CACHE_DIR / f"scenes_{year}.csv"
        if meta.exists():
            df = pd.read_csv(meta)
            df["time"] = _utc_datetime_ns(df["time"])  # ns UTC — required for merge_asof
            frames.append(df)
    if not frames:
        raise FileNotFoundError("No scenes_*.csv — run Step 5 first.")
    return pd.concat(frames, ignore_index=True).sort_values("time").reset_index(drop=True)


def tag_scenes_with_tide(scenes: pd.DataFrame, tide_series: pd.Series) -> pd.DataFrame:
    """Nearest FES2022 tide height (±30 min) for each scene acquisition time."""
    tide_df = tide_series.rename("tide_height").reset_index()
    tide_df.columns = ["time", "tide_height"]
    tide_df["time"] = _utc_datetime_ns(tide_df["time"])

    scenes = scenes.copy()
    scenes["time"] = _utc_datetime_ns(scenes["time"])

    tagged = pd.merge_asof(
        scenes.sort_values("time"),
        tide_df.sort_values("time"),
        on="time",
        direction="nearest",
        tolerance=pd.Timedelta("30min"),
    )
    return tagged.sort_values("time").reset_index(drop=True)


print("Helpers defined.")


## Step 5 — Build the NDWI time series (year caches)

We download **one calendar year at a time** and store:

- `cache/elevation_<site>/ndwi_<year>.zarr` — NDWI stack for that year
- `cache/elevation_<site>/scenes_<year>.csv` — scene times for Step 6

On re-run, existing years are skipped instantly (`OVERWRITE = False`).
All years are concatenated into one lazy stack passed to `elevation()`.


In [ ]:
parts, scene_counts = [], []

for year in YEARS:
    path, n = cache_year(year, overwrite=OVERWRITE)
    scene_counts.append(n if n == n else 0)
    if path is not None:
        parts.append(xr.open_zarr(path))  # lazy — not loaded into RAM yet

n_scenes = int(sum(scene_counts))
print(f"\nTotal clear-scene passes in window: {n_scenes}")

if not parts:
    raise SystemExit("No Landsat data — widen dates, AOI, or MAX_CLOUD.")

# Single multi-year stack sorted chronologically
sat_ds = xr.concat(parts, dim="time").sortby("time")
print("Combined stack:", dict(sat_ds.sizes))

# Same [OK] / [WARNING] pattern as pages 3–5
flags = []
if n_scenes < MIN_SCENES_RELIABLE:
    flags.append(
        f"FEW SCENES ({n_scenes} < {MIN_SCENES_RELIABLE}) — elevation fit may be noisy"
    )
status = "OK" if not flags else "WARNING"
print("=" * 60)
print(f"[{status}] Scene count — {SITE_NAME}")
for f in flags:
    print(f"  ⚠  {f}")
if not flags:
    print("  ✓  Enough scenes for a stable elevation fit.")
print("=" * 60)


### Interpreting Step 5

| Output | Good sign | Warning sign |
|---|---|---|
| **Scenes per year** | Several per year | Zero for a year → gap in stack |
| **Total scenes** | ≥ 20 | < 20 → noisy fits; extend window |
| **Cache hit** | Instant reload | First run downloads GB of data |
| **Stack shape** | `time` ≈ total scenes | `time` = 0 → check STAC filters |

If WARNING: extend `END`, raise `MAX_CLOUD`, or revisit page 3 tidal feasibility.


## Step 6 — Tide × satellite acquisitions

Same style as **page 3**: grey **FES2022 curve** (30 min) with **Landsat overpasses**
as coloured dots at the tide height that `elevation()` will use.

| Line / band | Meaning |
|---|---|
| **Red/blue dashed** | HAT and LAT — full astronomical range from FES2022 |
| **Red/blue dotted** | HOT and LOT — highest/lowest tide at an actual scene |
| **Shaded bands** | Tidal range **not** sampled by satellites |

Wide shaded bands mean the algorithm must **extrapolate** beyond observed tide levels.


In [ ]:
# Continuous FES2022 series (cached from pages 3–4 if same window)
tide_series = load_or_compute_tides(
    lon=LON, lat=LAT,
    start=START, end=END,
    site_name=SITE_NAME,
    tide_model=TIDE_MODEL,
    tide_dir=TIDE_DIR,
    overwrite=False,
)

# Tag each Landsat scene with nearest model tide height
scenes_df = tag_scenes_with_tide(load_scene_table(), tide_series)
obs = scenes_df["tide_height"].dropna()
if obs.empty:
    raise ValueError("No scenes with tide heights — check FES2022 / tide_models.")

# Reference levels (same definitions as page 3)
hat = float(tide_series.max())   # Highest Astronomical Tide (model)
lat_ = float(tide_series.min())  # Lowest Astronomical Tide (model)
hot = float(obs.max())           # Highest Observed Tide (satellite)
lot = float(obs.min())           # Lowest Observed Tide (satellite)
tr = hat - lat_

offset_high_m = hat - hot
offset_low_m = lot - lat_
offset_high_pct = offset_high_m / tr if tr > 0 else float("nan")
offset_low_pct = offset_low_m / tr if tr > 0 else float("nan")

print(f"HAT {hat:+.3f} m  |  LAT {lat_:+.3f} m  |  range {tr:.3f} m")
print(f"HOT {hot:+.3f} m  |  LOT {lot:+.3f} m")
print(f"Unsampled above HOT: {offset_high_m:.3f} m ({offset_high_pct:.0%} of range)")
print(f"Unsampled below LOT: {offset_low_m:.3f} m ({offset_low_pct:.0%} of range)")

# --- plot ---
platform_colors = {
    "landsat-5": "#8c510a",
    "landsat-7": "#bf812d",
    "landsat-8": "#35978f",
    "landsat-9": "#01665e",
}

fig, ax = plt.subplots(figsize=(14, 5))

# Background tide curve
tide_ts = tide_series.copy()
tide_ts.index = pd.to_datetime(tide_ts.index, utc=True)
ax.plot(
    tide_ts.index, tide_ts.values,
    color="#cccccc", lw=0.7, zorder=0, label="FES2022 tide (30 min)",
)

# One colour per Landsat platform
for platform in scenes_df["platform"].unique():
    sub = scenes_df.loc[scenes_df["platform"] == platform]
    ax.scatter(
        sub["time"], sub["tide_height"], s=22, alpha=0.8,
        color=platform_colors.get(platform, "grey"),
        label=platform.replace("landsat-", "Landsat "), zorder=2,
    )

# HAT/LAT (model extremes) and HOT/LOT (observed extremes)
ax.axhline(hat, color="#d7301f", lw=1.0, ls="--", label=f"HAT {hat:+.2f} m")
ax.axhline(lat_, color="#4575b4", lw=1.0, ls="--", label=f"LAT {lat_:+.2f} m")
ax.axhline(hot, color="#d7301f", lw=0.8, ls=":", alpha=0.9)
ax.axhline(lot, color="#4575b4", lw=0.8, ls=":", alpha=0.9)

# Shaded unsampled bands + annotation boxes (page 3 style)
x_annot = tide_ts.index.max()
for y0, y1, col, txt in [
    (hot, hat, "#d7301f", f"HAT offset\n{offset_high_m:.2f} m ({offset_high_pct:.0%})"),
    (lat_, lot, "#4575b4", f"LAT offset\n{offset_low_m:.2f} m ({offset_low_pct:.0%})"),
]:
    ax.fill_between(tide_ts.index, y0, y1, color=col, alpha=0.12, zorder=1)
    ax.annotate(
        txt, xy=(x_annot, (y0 + y1) / 2), xytext=(12, 0),
        textcoords="offset points", va="center", ha="left", fontsize=9, color=col,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=col, alpha=0.9),
    )

ax.set_xlabel("Date")
ax.set_ylabel("Tide height (m, MSL)")
ax.set_title(
    f"{SITE_NAME} — Landsat sampling vs FES2022 "
    f"({START[:4]}–{END[:4]}, n={len(scenes_df)})"
)
ax.legend(fontsize=8, ncol=2, loc="upper left")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
fig.tight_layout()
fig.savefig(f"{SITE_NAME}_tidal_sampling_elevation.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {SITE_NAME}_tidal_sampling_elevation.png")


### Interpreting Step 6

| Pattern | Meaning | Action |
|---|---|---|
| **Dots spread along grey curve** | Good tidal coverage | Proceed to Step 7 |
| **Large red/blue shaded bands** | HAT/LAT extremes rarely observed | Expect higher uncertainty at range edges |
| **Clusters only mid-range** | Poor sampling | Extend `END` or check page 3 |
| **Gaps in time** | Normal (~16 d Landsat revisit) | Multi-year window compensates |

The tide tags in this plot are the **same** values passed to `elevation()` in Step 7.


## Step 7 — Run `elevation()`

DEA Intertidal fits, for every pixel, the tide height at which NDWI switches from wet
to dry. That transition height is the **elevation** estimate.

Internally the function:

1. Tags each NDWI observation with FES2022 tide height
2. Builds a wet/dry vs tide curve per pixel
3. Returns elevation, uncertainty, and QA layers

> **Note:** `sat_ds.compute()` loads the full stack into RAM. For a small AOI this
> takes a few minutes; enlarge `DELTA` cautiously.


In [ ]:
print("Computing elevation (loading NDWI stack into memory) ...")
sat_ds = sat_ds.compute()

# elevation() expects an xarray Dataset with an 'ndwi' variable
result = elevation(
    sat_ds[["ndwi"]],
    tide_model=TIDE_MODEL,
    tide_model_dir=TIDE_DIR,
)
ds_elev = result[0] if isinstance(result, tuple) else result

# Keep the layers we use in this tutorial
keep = [v for v in [
    "elevation", "elevation_uncertainty",
    "qa_ndwi_freq", "qa_ndwi_corr", "qa_count_clear",
] if v in ds_elev]
ds_elev = ds_elev[keep]

print("Variables:", list(ds_elev.data_vars))
elev = ds_elev["elevation"]
print(f"Elevation range: {float(elev.min(skipna=True)):.2f} to {float(elev.max(skipna=True)):.2f} m")
print(f"Valid elevation pixels: {int(elev.notnull().sum()):,}")


### Interpreting Step 7

| Output line | Good sign | Warning sign |
|---|---|---|
| **Variables listed** | `elevation` + `elevation_uncertainty` present | Missing uncertainty → check DEA version |
| **Elevation range** | Matches local tidal range (roughly LAT–HAT) | All NaN → no intertidal pixels in AOI |
| **Valid pixel count** | Thousands+ for ~8 km box | Very few → widen AOI or date window |

If valid pixels ≪ wet/dry contrast on page 5 NDWI maps, revisit Step 6 sampling before publishing maps.


## Step 8 — Elevation map + uncertainty

Final deliverable: a **two-panel figure** — sediment height and per-pixel uncertainty.

- **Left:** elevation (m, MSL) — viridis colour scale
- **Right:** uncertainty (m) — lower values = more confident fit


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Panel 1: elevation ---
ds_elev["elevation"].plot.imshow(
    ax=axes[0], cmap="viridis",
    cbar_kwargs={"label": "Elevation (m, MSL)", "shrink": 0.75},
)
axes[0].set_title("Intertidal elevation")
axes[0].set_aspect("equal")
axes[0].set_xticks([])
axes[0].set_yticks([])

# --- Panel 2: uncertainty ---
if "elevation_uncertainty" in ds_elev:
    ds_elev["elevation_uncertainty"].plot.imshow(
        ax=axes[1], cmap="magma",
        cbar_kwargs={"label": "Uncertainty (m)", "shrink": 0.75},
    )
    axes[1].set_title("Per-pixel uncertainty")
else:
    axes[1].text(
        0.5, 0.5, "No uncertainty layer",
        ha="center", va="center", transform=axes[1].transAxes,
    )
axes[1].set_aspect("equal")
axes[1].set_xticks([])
axes[1].set_yticks([])

fig.suptitle(
    f"{SITE_NAME} — DEA intertidal elevation {START[:4]}–{END[:4]} (n={n_scenes} scenes)",
    fontsize=13, y=1.02,
)
fig.tight_layout()
fig.savefig(f"{SITE_NAME}_elevation_map.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {SITE_NAME}_elevation_map.png")


### Interpreting Step 8

| Panel | What to look for | Red flag |
|---|---|---|
| **Elevation** | Smooth gradient: channels low → upper flat high | Salt-and-pepper noise |
| **Uncertainty** | Low on central flats | High everywhere → poor sampling (Step 6) |

**Ask yourself:**

1. Does the elevation pattern match the flats you saw on pages 5–6?
2. Is uncertainty **low** where you need heights for your research question?
3. High uncertainty at the flat edges only → normal; high uncertainty on the flat centre → extend the date window.


## Step 9 — Quality assurance maps

DEA Intertidal returns **QA layers** alongside elevation. Together they show *where* the
fit is trustworthy and *why* some pixels fail.

| Layer | Meaning | What to look for |
|---|---|---|
| `elevation` | Sediment height (m, MSL) | Smooth intertidal gradient |
| `elevation_uncertainty` | Fit uncertainty (m) | Low on central flats |
| `qa_ndwi_corr` | Correlation between NDWI and tide height | **Positive** on intertidal pixels (wet at high tide) |
| `qa_ndwi_freq` | Fraction of clear observations classified wet | ~0 on upper flat, ~1 in channels |
| `qa_count_clear` | Number of clear satellite observations | Higher → more stable fit |

Same five-panel layout as the
[DEA development notebook](https://github.com/GeoscienceAustralia/dea-intertidal/blob/develop/notebooks/development/Intertidal_elevation.ipynb).


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))

layers = [
    ("elevation", "Elevation (m, MSL)", {"cmap": "viridis"}),
    ("elevation_uncertainty", "Uncertainty (m)", {"cmap": "inferno", "vmin": 0, "vmax": 0.5}),
    ("qa_ndwi_corr", "NDWI–tide correlation", {"cmap": "RdBu", "vmin": -0.7, "vmax": 0.7}),
    ("qa_ndwi_freq", "Wet obs. frequency", {"cmap": "Blues", "vmin": 0, "vmax": 1}),
    ("qa_count_clear", "Clear obs. count", {"cmap": "Greys"}),
]

for ax, (var, title, plot_kw) in zip(axes, layers):
    ax.set_title(title, fontsize=10)
    if var not in ds_elev:
        ax.text(0.5, 0.5, f"No {var}", ha="center", va="center", transform=ax.transAxes)
    else:
        ds_elev[var].plot.imshow(ax=ax, add_colorbar=True, **plot_kw)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

fig.suptitle(
    f"{SITE_NAME} — elevation + QA layers {START[:4]}–{END[:4]}",
    fontsize=13, y=1.05,
)
fig.tight_layout()
fig.savefig(f"{SITE_NAME}_elevation_qa.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {SITE_NAME}_elevation_qa.png")


### Interpreting Step 9

| Panel | Good sign | Warning sign |
|---|---|---|
| **NDWI–tide correlation** | Positive on intertidal flats | Near zero everywhere → poor tidal sampling (Step 6) |
| **Wet frequency** | Gradient from dry upper flat to wet channels | Uniform values → pixel never toggles wet/dry |
| **Clear obs. count** | Dozens+ on flats | Very low counts → noisy or missing elevation |

Use these maps to **mask** unreliable pixels before exporting statistics — e.g. keep pixels
with `qa_count_clear ≥ 20` and `elevation_uncertainty` below a chosen threshold.


## Step 10 — Save outputs

| File | Format | Use |
|---|---|---|
| `*_elevation_*.nc` | NetCDF | Full dataset + QA layers (Python/xarray) |
| `*_elevation_*.tif` | GeoTIFF | Elevation band in QGIS / ArcGIS |


In [ ]:
out_nc = f"{SITE_NAME}_elevation_{START[:4]}_{END[:4]}.nc"
ds_elev.to_netcdf(out_nc)
print(f"Wrote: {out_nc}")

# Single-band GeoTIFF for GIS (requires rioxarray + CRS on the array)
try:
    tif = f"{SITE_NAME}_elevation_{START[:4]}_{END[:4]}.tif"
    ds_elev["elevation"].rio.to_raster(tif)
    print(f"Wrote: {tif}")
except Exception as e:
    print("GeoTIFF export skipped:", e)


## Step 11 — What you have produced (recap)

In this notebook you built one elevation story in six layers:

1. **Year caches (Step 5)** — Landsat NDWI per calendar year (`ndwi_<year>.zarr`)
2. **Tidal sampling (Step 6)** — Landsat dots on the FES2022 curve (HAT/LAT/HOT/LOT)
3. **Elevation fit (Step 7)** — DEA `elevation()` with FES2022 tide tags
4. **Maps (Step 8)** — height + uncertainty side by side
5. **QA maps (Step 9)** — correlation, wet frequency, clear count
6. **Exports (Step 10)** — NetCDF + GeoTIFF for downstream use

### Output files (replace `SITE_NAME` with your site label)

| File | Content |
|---|---|
| `SITE_NAME_tidal_sampling_elevation.png` | Tide curve + Landsat scene dots |
| `SITE_NAME_elevation_map.png` | Elevation + uncertainty (2 panels) |
| `SITE_NAME_elevation_qa.png` | Elevation + full QA stack (5 panels) |
| `SITE_NAME_elevation_YYYY_YYYY.nc` | Full elevation dataset + QA |
| `SITE_NAME_elevation_YYYY_YYYY.tif` | Elevation GeoTIFF for GIS |

### Caveats (state in your methods section)

- **Tidal sampling bias** — Landsat sun-synchronous orbits miss some tide phases (Step 6)
- **Datum** — heights are MSL via FES2022; apply NAP offset if comparing to RWS (page 4)
- **Morphology** — assumes stable bathymetry over the image window
- **One AOI** — this notebook maps **one box**; batch estuary processing is a separate workflow

**You finished the six tutorial pages.** Continue to **[Applications & next steps](07_applications.md)** for project ideas — time series, ecological monitoring, shapefile-based analysis — or optionally **[Cleanup](07_cleanup.md)** to remove the local install.

Setup → Connect → Tides → Validation → NDWI → **Elevation** → *your project*
